# 🚀 Project SETU — Cloud-Scale Dual-Hand Indian Sign Language (ISL) Training
### High-Accuracy Deep Learning Pipeline for 26-Class ISL Recognition (A–Z)
**Designed for Google Colab (Free Tesla T4 GPU) & Kaggle Notebooks**

---
### 📌 Why Train in the Cloud Instead of Locally?
1. **Computational Power:** Google Colab provides a free 16 GB Tesla T4 GPU, allowing large batch sizes (64–128) and faster epoch convergence.
2. **Authentic ISL Bimanual Geometry:** Indian Sign Language (ISL) is predominantly **two-handed** (e.g. B, D, P, Q, T). This notebook trains on dual-hand 126-D coordinate space.
3. **Geometric Data Augmentation:** Real-time 2D planar rotation (+-15°), coordinate jitter, and scaling prevent overfitting and ensure real-world camera invariance.
4. **Direct Export:** Generates `realsign_26class_mlp.pth` which drops straight into SETU without changing any backend code!

In [ ]:
# Step 1: Verify GPU & Install Dependencies
!nvidia-smi
!pip install -q torch torchvision mediapipe opencv-python matplotlib seaborn scikit-learn

In [ ]:
# Step 2: Import Libraries & Set Seeds
import os, sys, math, random, time, datetime
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import seaborn as sns

# Set deterministic seeds
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'🔥 Training engine active on: {device}')

In [ ]:
# Step 3: Dataset Ingestion
# Upload isl_master_clean_126d.npz directly from your project data/processed folder
from google.colab import files
print('Please upload isl_master_clean_126d.npz:')
uploaded = files.upload()

dataset_filename = list(uploaded.keys())[0]
data = np.load(dataset_filename, allow_pickle=True)
X = data['X'].astype(np.float32)
y = data['y'].astype(np.int64)
classes = list(data.get('classes', [chr(ord('A') + i) for i in range(26)]))
print(f'✅ Successfully loaded {len(X)} samples across {len(classes)} classes. Feature dimension: {X.shape[1]}')

In [ ]:
# Step 4: Real-Time Geometric Data Augmenter
class DualHandLandmarkAugmenter:
    def __init__(self, noise_std=0.012, rotation_deg=14.0, scale_range=(0.90, 1.10)):
        self.noise_std = noise_std
        self.max_rot_rad = math.radians(rotation_deg)
        self.scale_range = scale_range

    def __call__(self, feat_126):
        out = feat_126.copy().reshape(-1, 3)
        if random.random() < 0.65:
            angle = random.uniform(-self.max_rot_rad, self.max_rot_rad)
            cos_a, sin_a = math.cos(angle), math.sin(angle)
            rot_matrix = np.array([[cos_a, -sin_a], [sin_a, cos_a]], dtype=np.float32)
            for wrist_idx, hand_slice in [(0, slice(0, 21)), (21, slice(21, 42))]:
                pts = out[hand_slice]
                if np.std(pts) > 1e-4:
                    wrist = pts[0, :2].copy()
                    out[hand_slice, :2] = ((pts[:, :2] - wrist) @ rot_matrix.T) + wrist
        if random.random() < 0.60:
            scale = random.uniform(*self.scale_range)
            for hand_slice in [slice(0, 21), slice(21, 42)]:
                if np.std(out[hand_slice]) > 1e-4:
                    wrist = out[hand_slice][0, :2].copy()
                    out[hand_slice, :2] = (out[hand_slice, :2] - wrist) * scale + wrist
        if random.random() < 0.70:
            noise = np.random.normal(0.0, self.noise_std, out.shape).astype(np.float32)
            noise[0] = 0.0
            noise[21] = 0.0
            out += noise
        return out.flatten().astype(np.float32)

class ISLDataset(Dataset):
    def __init__(self, X, y, is_train=True, augmenter=None):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
        self.augmenter = augmenter if (is_train and augmenter) else None
    def __len__(self):
        return len(self.y)
    def __getitem__(self, idx):
        feat = self.X[idx].numpy()
        if self.augmenter:
            feat = self.augmenter(feat)
        return torch.tensor(feat, dtype=torch.float32), self.y[idx]

In [ ]:
# Step 5: Dataset Split & DataLoaders
num_samples = len(X)
indices = np.arange(num_samples)
np.random.shuffle(indices)
n_train = int(0.80 * num_samples)
n_val = int(0.10 * num_samples)

train_ds = ISLDataset(X[indices[:n_train]], y[indices[:n_train]], is_train=True, augmenter=DualHandLandmarkAugmenter())
val_ds = ISLDataset(X[indices[n_train:n_train+n_val]], y[indices[n_train:n_train+n_val]], is_train=False)
test_ds = ISLDataset(X[indices[n_train+n_val:]], y[indices[n_train+n_val:]], is_train=False)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False)
print(f'Data split: {len(train_ds)} train, {len(val_ds)} val, {len(test_ds)} test')

In [ ]:
# Step 6: Define Neural Architecture (SETU Production RealSignMLP Drop-in)
class RealSignMLP(nn.Module):
    def __init__(self, input_dim=126, hidden1=128, hidden2=64, num_classes=26, dropout=0.20, use_batchnorm=True):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden1),
            nn.BatchNorm1d(hidden1),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden1, hidden2),
            nn.BatchNorm1d(hidden2),
            nn.ReLU(),
            nn.Linear(hidden2, num_classes),
        )
    def forward(self, x):
        return self.net(x)

model = RealSignMLP(input_dim=X.shape[1], hidden1=128, hidden2=64, num_classes=len(classes)).to(device)
print(model)

In [ ]:
# Step 7: Train Model with Cosine Annealing Learning Rate
epochs = 60
criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
optimizer = torch.optim.AdamW(model.parameters(), lr=1.5e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-5)

best_val_acc = 0.0
best_weights = None

for epoch in range(1, epochs + 1):
    model.train()
    correct, total = 0, 0
    for feats, targets in train_loader:
        feats, targets = feats.to(device), targets.to(device)
        optimizer.zero_grad()
        out = model(feats)
        loss = criterion(out, targets)
        loss.backward()
        optimizer.step()
        correct += (out.argmax(1) == targets).sum().item()
        total += len(targets)
    scheduler.step()
    
    model.eval()
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for feats, targets in val_loader:
            feats, targets = feats.to(device), targets.to(device)
            out = model(feats)
            val_correct += (out.argmax(1) == targets).sum().item()
            val_total += len(targets)
    
    val_acc = val_correct / max(val_total, 1)
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_weights = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    
    if epoch % 5 == 0 or epoch == epochs:
        print(f'Epoch [{epoch:2d}/{epochs:2d}] | Train Acc: {correct/total*100:5.2f}% | Val Acc: {val_acc*100:5.2f}% (Best: {best_val_acc*100:5.2f}%)')

In [ ]:
# Step 8: Comprehensive Evaluation & Mentor Metrics
model.load_state_dict(best_weights)
model.eval()
all_preds, all_targets = [], []
with torch.no_grad():
    for feats, targets in test_loader:
        feats = feats.to(device)
        all_preds.extend(model(feats).argmax(1).cpu().tolist())
        all_targets.extend(targets.tolist())

all_preds = np.array(all_preds)
all_targets = np.array(all_targets)
test_acc = np.mean(all_preds == all_targets)
print(f'🎯 Final Test Accuracy across all 26 ISL Classes: {test_acc*100:.2f}%')

# Plot Confusion Matrix
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(all_targets, all_preds)
plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes)
plt.title(f'SETU ISL 26-Class Confusion Matrix (Test Accuracy: {test_acc*100:.2f}%)')
plt.xlabel('Predicted Letter')
plt.ylabel('Ground Truth Letter')
plt.show()

In [ ]:
# Step 9: Export Checkpoint & Download to PC
checkpoint = {
    'model_state_dict': best_weights,
    'classes': classes,
    'test_accuracy': float(test_acc),
    'trained_at': datetime.datetime.now().isoformat(),
    'architecture': {
        'input_dim': X.shape[1],
        'hidden1': 128,
        'hidden2': 64,
        'num_classes': len(classes),
        'use_batchnorm': True,
        'dropout': 0.20,
    },
}
torch.save(checkpoint, 'realsign_26class_mlp.pth')
print('✅ Checkpoint saved: realsign_26class_mlp.pth')

# Auto-download file directly to your laptop
files.download('realsign_26class_mlp.pth')
print('🚀 Drop this downloaded file into: models/weights/realsign_26class_mlp.pth')